# TMR4240 Project – Vessel Plant Model: Equations and Matrices

This notebook collects the model equations used in the project and prints the **numerical
matrices of the R/V Gunnerus 3-DOF model** directly from the `mcsimpy` vessel data, so you
can use the exact values in your report and controller design.

**References:** Sørensen, *Marine Control Systems* (Ch. 6–7); Fossen, *Handbook of Marine
Craft Hydrodynamics and Motion Control* (Ch. 2, 6–7).

Run the cells top to bottom. Requirements are the same as for the project template
(`pip install -e .` in the repository root).

## 1. Coordinate Frames

Two frames describe the vessel motion:

- **NED frame** $\{n\}$ (Earth-fixed): $x_n$ North, $y_n$ East, $z_n$ Down. Used for position and heading.
- **Body frame** $\{b\}$ (vessel-fixed): $x_b$ toward the bow (surge), $y_b$ to starboard (sway), $z_b$ down.

Both frames are right-handed. The yaw angle $\psi$ is measured **clockwise from North** to the bow direction (positive rotation about the downward $z$ axis).

## 2. Notation: Six DOF and the Three-DOF Reduction

Standard six-DOF marine craft vectors:

$$
\eta_6 = [x,\, y,\, z,\, \phi,\, \theta,\, \psi]^\top \;(\text{NED}), \qquad
\nu_6 = [u,\, v,\, w,\, p,\, q,\, r]^\top \;(\text{body}), \qquad
\tau_6 = [X,\, Y,\, Z,\, K,\, M,\, N]^\top \;(\text{body}).
$$

This project uses the horizontal-plane three-DOF model (heave, roll, pitch neglected):

$$
\eta = [N,\, E,\, \psi]^\top \;(\text{NED}), \qquad
\nu = [u,\, v,\, r]^\top \;(\text{body}), \qquad
\tau = [X,\, Y,\, N_z]^\top \;(\text{body}).
$$

Here $N_z$ denotes the yaw moment (written with a subscript to avoid confusion with the North position $N$). The dynamics compute accelerations in **body** axes; the position is updated in **NED**.

## 3. Kinematics

$$
\dot\eta = J(\psi)\,\nu, \qquad
J(\psi) =
\begin{bmatrix}
\cos\psi & -\sin\psi & 0 \\
\sin\psi & \cos\psi & 0 \\
0 & 0 & 1
\end{bmatrix}.
$$

Scalar form:

$$
\dot N = u\cos\psi - v\sin\psi, \qquad
\dot E = u\sin\psi + v\cos\psi, \qquad
\dot\psi = r.
$$

Since $J(\psi)$ is a rotation matrix, $J^{-1}(\psi) = J^\top(\psi)$: it converts NED vectors to the body frame (e.g. a NED position error into body-frame force directions, or the NED current vector into body axes).

Heading errors must always be wrapped to $(-\pi, \pi]$:

$$
\mathrm{wrap}(\alpha) = \operatorname{atan2}(\sin\alpha, \cos\alpha).
$$

In the template these live in `simulation/utils.py` as `Rz(psi)` and `wrap_angle_pi(angle)`:

In [1]:
import sys
sys.path.insert(0, "..")  # allow running without installing the project package

import numpy as np
from simulation.utils import Rz, wrap_angle_pi

psi = np.deg2rad(30.0)
J = Rz(psi)
print("J(30 deg) =\n", np.round(J, 4))
print("\nJ^T J = I:", np.allclose(J.T @ J, np.eye(3)))
print("wrap(190 deg) =", np.rad2deg(wrap_angle_pi(np.deg2rad(190.0))), "deg")

J(30 deg) =
 [[ 0.866 -0.5    0.   ]
 [ 0.5    0.866  0.   ]
 [ 0.     0.     1.   ]]

J^T J = I: True
wrap(190 deg) = -170.0 deg


## 4. Process Plant Model (simulation / "truth" model)

General six-DOF marine craft dynamics:

$$
M\dot\nu_6 + C(\nu_6)\nu_6 + D(\nu_r)\nu_r + g(\eta_6) = \tau_6 .
$$

For horizontal-plane three-DOF motion there are no hydrostatic restoring forces in surge, sway, and yaw, so $g \equiv 0$ and the model reduces to

$$
M\dot\nu + C_{RB}(\nu)\nu + C_A(\nu_r)\nu_r + D(\nu_r)\nu_r = \tau,
\qquad M = M_{RB} + M_A .
$$

**Current** enters through the relative velocity. With current speed $U_c$ and direction $\beta_c$ (convention: **towards** — the direction the current flows *to*):

$$
\nu_c^n = [V_{c,N},\, V_{c,E},\, 0]^\top = U_c\,[\cos\beta_c,\, \sin\beta_c,\, 0]^\top, \qquad
\nu_c^b = J^\top(\psi)\,\nu_c^n, \qquad
\nu_r = \nu - \nu_c^b .
$$

**Damping** in the Gunnerus model is linear plus velocity-dependent terms:

$$
D(\nu_r) = D_l + D_u\,|u_r| + D_v\,|v_r| + D_r\,|r| .
$$

The next cell loads the numerical matrices from the `mcsimpy` vessel data file
(`parV_RVG3DOF.pkl`) — the same file the simulator uses:

In [2]:
import pickle
from importlib.resources import files
from IPython.display import Math, display

pkl = files("mcsimpy.vessel_data.gunnerus") / "parV_RVG3DOF.pkl"
with open(str(pkl), "rb") as f:
    data = pickle.load(f)

M_RB, M_A = data["Mrb"], data["Ma"]
D_l, D_u, D_v, D_r = data["Dl"], data["Du"], data["Dv"], data["Dr"]
M = M_RB + M_A


def _fmt(x):
    # format a number for LaTeX, scientific notation for large/small values
    if x == 0:
        return "0"
    if abs(x) >= 1e4 or abs(x) < 1e-2:
        m, e = f"{x:.4e}".split("e")
        return rf"{float(m):.4g} \times 10^{{{int(e)}}}"
    return f"{x:.5g}"


def bmatrix(A):
    A = np.atleast_2d(A)
    rows = [" & ".join(_fmt(x) for x in row) for row in A]
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"


def show(lhs, A):
    display(Math(lhs + " = " + bmatrix(A)))


show(r"M_{RB}", M_RB)
show(r"M_A", M_A)
show(r"M = M_{RB} + M_A", M)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [3]:
show(r"D_l", D_l)
show(r"D_u", D_u)
show(r"D_v", D_v)
show(r"D_r", D_r)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Coriolis matrices — computed, not stored

The process plant is **not** just mass and damping. The full set of terms is

$$
\underbrace{M\dot\nu}_{\text{inertia}}
+ \underbrace{C_{RB}(\nu)\,\nu + C_A(\nu_r)\,\nu_r}_{\text{Coriolis--centripetal}}
+ \underbrace{D(\nu_r)\,\nu_r}_{\text{damping}}
+ \underbrace{g(\eta)}_{=\,0 \text{ in the horizontal plane}}
= \tau .
$$

There is simply no stored $C$ matrix in the vessel data, because for 3-DOF the Coriolis matrix is
**built from the entries of the mass matrix** at every time step (mcsimpy's `Cor3` function):

$$
C(\nu, M) =
\begin{bmatrix}
0 & 0 & -M_{22}\,v - \tfrac{1}{2}(M_{23} + M_{32})\,r \\
0 & 0 & M_{11}\,u \\
M_{22}\,v + \tfrac{1}{2}(M_{23} + M_{32})\,r & -M_{11}\,u & 0
\end{bmatrix},
$$

evaluated twice per step: $C_{RB} = C(\nu, M_{RB})$ acting on $\nu$, and $C_A = C(\nu_r, M_A)$ acting
on $\nu_r$. At $\nu = 0$ both vanish — which is why they can be dropped from the control plant. The
next cell evaluates them at an example velocity so you can see their size relative to the damping
terms:

In [4]:
def Cor3(nu, M):
    return np.array(
        [
            [0, 0, -M[1, 1] * nu[1] - 0.5 * (M[1, 2] + M[2, 1]) * nu[2]],
            [0, 0, M[0, 0] * nu[0]],
            [
                M[1, 1] * nu[1] + 0.5 * (M[1, 2] + M[2, 1]) * nu[2],
                -M[0, 0] * nu[0],
                0,
            ],
        ]
    )

nu_ex = np.array([1.0, 0.3, 0.05])   # example: 1 m/s surge, 0.3 m/s sway, 0.05 rad/s yaw
show(r"\nu_{\text{ex}}", nu_ex.reshape(-1, 1))
show(r"C_{RB}(\nu_{\text{ex}})", Cor3(nu_ex, M_RB))
show(r"C_A(\nu_{\text{ex}})", Cor3(nu_ex, M_A))
show(r"D(\nu_{\text{ex}}) = D_l + D_u|u| + D_v|v| + D_r|r|",
     D_l + D_u * abs(nu_ex[0]) + D_v * abs(nu_ex[1]) + D_r * abs(nu_ex[2]))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Units: mass entries in $\mathrm{kg}$, mass–yaw coupling in $\mathrm{kg\,m}$, yaw inertia in $\mathrm{kg\,m^2}$; damping entries are the corresponding force/moment per unit (relative) velocity.

## 5. Form Used for Numerical Integration

The simulator state is $x = [\eta^\top,\ \nu^\top]^\top \in \mathbb{R}^6$. Rearranging the plant equation:

$$
\dot\nu = M^{-1}\big(\tau - C_{RB}(\nu)\nu - C_A(\nu_r)\nu_r - D(\nu_r)\nu_r + M_A\dot\nu_c^b\big),
\qquad
\dot x =
\begin{bmatrix}
J(\psi)\,\nu \\
M^{-1}(\,\cdots)
\end{bmatrix}.
$$

The term $M_A\dot\nu_c^b$ (with $\dot\nu_c^b$ the body-frame rate of change of the current velocity) appears because the equation is written in $\nu$ rather than $\nu_r$; the equivalent relative-velocity form is $M\dot\nu_r + C(\nu_r)\nu_r + D(\nu_r)\nu_r = \tau$.

The base vessel class integrates $\dot x$ with either method:

**Forward Euler:** $\;x_{k+1} = x_k + h\, f(t_k, x_k)$.

**Runge–Kutta 4:**

$$
k_1 = f(t_k, x_k), \quad
k_2 = f\!\left(t_k + \tfrac{h}{2},\, x_k + \tfrac{h}{2} k_1\right), \quad
k_3 = f\!\left(t_k + \tfrac{h}{2},\, x_k + \tfrac{h}{2} k_2\right), \quad
k_4 = f(t_k + h,\, x_k + h k_3),
$$

$$
x_{k+1} = x_k + \tfrac{h}{6}\,(k_1 + 2k_2 + 2k_3 + k_4).
$$

Remember to state the chosen time step $h$ and integration method in your report.

## 6. Control Plant Model (design model)

For controller design a simplified linear model is used — low-speed motion, small deviations
around the operating point, **linear damping only**, Coriolis/centripetal terms neglected, no
restoring forces in the horizontal plane:

$$
\dot\eta = J(\psi)\,\nu, \qquad
M_3\,\dot\nu + D_3\,\nu = \tau_c .
$$

With the Gunnerus data the natural choice is

$$
M_3 = M_{RB} + M_A, \qquad D_3 = D_l ,
$$

i.e. the full inertia matrix and the **linear part** of the damping. $\eta$ is in NED, $\nu$ and $\tau_c$ in the body frame.

In [5]:
M3 = M_RB + M_A
D3 = D_l

show(r"M_3", M3)
show(r"D_3", D3)
show(r"M_3^{-1}", np.linalg.inv(M3))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

If a state-space method (LQR, pole placement) is used, define $x_c = [\eta^\top,\ \nu^\top]^\top$ and linearize about the setpoint heading $\psi_0$ (often $\psi_0 = 0$ so that $J \approx I$):

$$
\dot x_c = A_c x_c + B_c \tau_c, \qquad y_c = C_c x_c,
$$

$$
A_c =
\begin{bmatrix}
0_{3\times3} & J(\psi_0) \\
0_{3\times3} & -M_3^{-1} D_3
\end{bmatrix},
\qquad
B_c =
\begin{bmatrix}
0_{3\times3} \\
M_3^{-1}
\end{bmatrix},
\qquad
C_c = \begin{bmatrix} I_{3\times3} & 0_{3\times3} \end{bmatrix}.
$$

In [6]:
psi0 = 0.0
A_c = np.block([
    [np.zeros((3, 3)), Rz(psi0)],
    [np.zeros((3, 3)), -np.linalg.inv(M3) @ D3],
])
B_c = np.vstack([np.zeros((3, 3)), np.linalg.inv(M3)])
C_c = np.hstack([np.eye(3), np.zeros((3, 3))])

show(r"A_c", A_c)
show(r"B_c", B_c)

poles = np.linalg.eigvals(A_c)
print("Open-loop poles:", np.round(poles, 6))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Open-loop poles: [ 0.      +0.j  0.      +0.j -0.030341+0.j -0.037408+0.j  0.      +0.j
 -0.00186 +0.j]


The three poles at the origin are the position/heading integrators; the remaining (stable, slow) poles come from $-M_3^{-1} D_3$ — this is why a DP vessel needs active feedback control.

## 7. Where Do These Matrices Come From? The Full Gunnerus Hydrodynamic Database

The six matrices above are already a strong simplification. The `mcsimpy` package also ships the
**full ShipX/Veres hydrodynamic database** for R/V Gunnerus (`vessel_2.json`), used by the 6-DOF DP
model. There, added mass and damping are not constant matrices at all but **matrix functions of wave
frequency $\omega$ and forward speed $U$**:

| Level | Model | Matrices |
|---|---|---|
| 1 | Hydrodynamic database (ShipX/Veres, 6-DOF) | $M_{RB}$ (6×6), $A(\omega, U)$, $B(\omega, U)$, $C$ — each 6×6 over 36 frequencies × 7 speeds — plus viscous $B_v$, force/motion RAOs, wave-drift coefficients |
| 2 | **Process plant** (3-DOF maneuvering, this project) | $M_{RB}$, $M_A$ (constant, zero-frequency), $D_l$, $D_u$, $D_v$, $D_r$; $C_{RB}, C_A$ computed from the mass matrices; $g = 0$ |
| 3 | **Control plant** (design model) | $M_3 = M_{RB} + M_A$, $D_3 = D_l$ |

Each level deliberately discards physics that does not matter for its purpose: the DP models use the
low-frequency, zero-speed limit of $A(\omega, U)$; the horizontal-plane reduction removes heave, roll,
pitch and with them all hydrostatic restoring; the control plant finally drops the nonlinear damping
and Coriolis terms, which are second order in velocity and small at DP speeds. This chain —
*database → process plant → control plant* — is exactly why the simplified control model is adequate
for controller design even though the simulation model is nonlinear, and why gain tuning against the
process plant remains iterative.

The next cell opens the level-1 database and shows its structure:

In [7]:
import json as _json

p6 = files("mcsimpy.vessel_data.gunnerus") / "vessel_2.json"
with open(str(p6)) as f:
    d6 = _json.load(f)

A6 = np.asarray(d6["A"])          # added mass A(omega, U)
B6 = np.asarray(d6["B"])          # potential (radiation) damping B(omega, U)
C6 = np.asarray(d6["C"])          # hydrostatic restoring
MRB6 = np.asarray(d6["MRB"])      # rigid-body mass, 6x6
Bv6 = np.asarray(d6["Bv"])        # viscous damping, 6x6
freqs = np.asarray(d6["freqs"])
vels = np.asarray(d6["velocities"])

print(f"A(omega, U): {A6.shape}  ->  6x6 matrix at each of {freqs.shape[0]} frequencies x {vels.shape[0]} speeds")
print(f"B(omega, U): {B6.shape}")
print(f"C:           {C6.shape}")
print(f"frequencies: {freqs.min():.3f} ... {freqs.max():.3f} rad/s")
print(f"speeds:      {vels} m/s")

# The 6-DOF DP model picks one slice: low frequency (index 30), zero speed (index 0).
A_dp = A6[:, :, 30, 0]
idx = np.ix_([0, 1, 5], [0, 1, 5])  # surge, sway, yaw sub-block

show(r"A(\omega_{30}, U{=}0)\big|_{u,v,r}\ \ (\text{database, DP slice})", A_dp[idx])
show(r"M_A\ \ (\text{3-DOF maneuvering model, used in this project})", M_A)

# The constant 6-DOF matrices of the database (order: surge, sway, heave, roll, pitch, yaw):
show(r"M_{RB}^{6DOF}", MRB6)
show(r"B_v^{6DOF}\ (\text{viscous damping})", Bv6)

# Hydrostatic restoring at the DP slice — nonzero ONLY in heave, roll, pitch,
# which is exactly why g = 0 in the horizontal-plane 3-DOF model:
show(r"C(\omega_{0}, U{=}0)\ (\text{restoring})", C6[:, :, 0, 0])

A(omega, U): (6, 6, 36, 7)  ->  6x6 matrix at each of 36 frequencies x 7 speeds
B(omega, U): (6, 6, 36, 7)
C:           (6, 6, 36, 7)
frequencies: 0.209 ... 4.189 rad/s
speeds:      [0.       1.028889 2.057778 3.086667 5.144444 6.173334 7.202222] m/s


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 8. Where the Equations Live in the Code

| Equation | Implementation |
|---|---|
| Kinematics $J(\psi)$, angle wrap | `simulation/utils.py` (`Rz`, `wrap_angle_pi`) |
| Process plant dynamics, relative velocity, damping, integration | `mcsimpy` `GunnerusManeuvering3DoF`, wrapped by `models/gunnerus_3dof.py` |
| Loop ordering: reference → control → allocation → actuator model (ideal in Part 1) → environment → vessel update | `simulation/simulation_part_1.py` |
| Model parameters ($M_{RB}$, $M_A$, $D_l$, $D_u$, $D_v$, $D_r$) | `mcsimpy` vessel data `parV_RVG3DOF.pkl` (loaded above) |

When writing the report, define every symbol before use and state the frame (NED or body) of every vector — the frame distinction is the core structural point of the model.